# ADAIL × MedMNIST RetinaMNIST

Notebook lokal untuk **class-incremental ordinal learning** pada RetinaMNIST.
Alurnya diselaraskan dengan notebook CCH5000:

- source ADAIL dibaca dari folder lokal `ADAIL` di direktori yang sama;
- skenario `3-1`: kelas `[0, 1, 2]`, lalu kelas `3`, lalu kelas `4`;
- parameter umum CCH: seed 42, 20 epoch/task, batch 32, input 224,
  dan exemplar memory 1.000;
- ordinal training aktif (`use_ordinal=True`, `lambda_ordinal=0.1`);
- evaluasi per task: accuracy, MAE, dan Quadratic Weighted Kappa (QWK);
- evaluasi ordinal akhir: MAE, QWK, akurasi tepat, akurasi toleransi ±1
  tingkat, confusion matrix, dan file prediksi;
- live log, partial history, checkpoint, summary CSV, dan ZIP output.

Struktur direktori yang diharapkan:

```text
direktori-eksperimen/
├── ADAIL/
│   └── src/adail/
├── cch old.ipynb
└── ADAIL_RetinaMNIST_Incremental_Ordinal_Local.ipynb
```

RetinaMNIST akan tersimpan otomatis di `data/medmnist`. Unduhan hanya
dibutuhkan pada eksekusi pertama.


In [ ]:
# ============================================================
# 1. Pengaturan eksperimen
# ============================================================
from pathlib import Path

# Bisa dijalankan dari root ADAIL, atau dari direktori yang berisi
# notebook CCH dan folder ADAIL.
NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "src" / "adail").is_dir():
    REPO_DIR = NOTEBOOK_DIR
else:
    REPO_DIR = NOTEBOOK_DIR / "ADAIL"
DATA_ROOT = NOTEBOOK_DIR / "data" / "medmnist"
OUTPUT_DIR = NOTEBOOK_DIR / "runs" / "adail_retinamnist_3_1"

DATASET_FLAG = "retinamnist"
SCENARIO = "3-1"
NUM_CLASSES = 5

# Gunakan split resmi RetinaMNIST. Validation tidak digabung ke train.
DOWNLOAD_DATA = True
MEDMNIST_SIZE = 224

# Konfigurasi umum mengikuti notebook CCH.
SEED = 42
EPOCHS_PER_TASK = 20
BATCH_SIZE = 32
IMAGE_SIZE = 224
NUM_WORKERS = 0
MEMORY_SIZE = 1000
LR = 0.01
USE_PRETRAINED_BACKBONE = True

# Konfigurasi ordinal.
USE_ORDINAL = True
LAMBDA_ORDINAL = 0.1

print("Notebook dir :", NOTEBOOK_DIR)
print("Repo ADAIL  :", REPO_DIR)
print("Data root   :", DATA_ROOT)
print("Output dir  :", OUTPUT_DIR)
print("Scenario    :", SCENARIO)


In [ ]:
# ============================================================
# 2. Import ADAIL lokal dan siapkan environment
# ============================================================
import importlib.util
import os
import random
import subprocess
import sys

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

SRC_DIR = REPO_DIR / "src"
if not (SRC_DIR / "adail").is_dir():
    raise FileNotFoundError(
        "Folder source ADAIL tidak ditemukan. Notebook harus dijalankan dari "
        f"direktori yang memiliki folder ADAIL. Path yang diperiksa: {SRC_DIR}"
    )
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Install hanya dependency dataset bila belum tersedia.
if importlib.util.find_spec("medmnist") is None:
    print("Package medmnist belum tersedia; melakukan instalasi...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "medmnist"],
        check=True,
    )

import adail
import medmnist
import numpy as np
import pandas as pd
import torch
import torchvision

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("ADAIL       :", getattr(adail, "__version__", "unknown"))
print("ADAIL file  :", adail.__file__)
print("MedMNIST    :", medmnist.__version__)
print("Torch       :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("Device      :", DEVICE)
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# 3. DataLoader aman + label memory konsisten
# ============================================================
import torch
from torch.utils.data import DataLoader as TorchDataLoader
from torch.utils.data._utils.collate import default_collate

import adail.trainer as adail_trainer
from adail.memory import MemoryDataset


def _label_to_int(label):
    if torch.is_tensor(label):
        return int(label.detach().cpu().reshape(-1)[0].item())
    return int(np.asarray(label).reshape(-1)[0])


def _safe_collate(batch):
    images, labels = zip(*batch)
    images = default_collate(list(images))
    labels = torch.tensor([_label_to_int(y) for y in labels], dtype=torch.long)
    return images, labels


def LocalSafeDataLoader(*args, **kwargs):
    kwargs["num_workers"] = NUM_WORKERS
    kwargs["persistent_workers"] = False
    kwargs["collate_fn"] = _safe_collate
    return TorchDataLoader(*args, **kwargs)


# ADAILTrainer menggunakan simbol DataLoader dari modul adail.trainer.
adail_trainer.DataLoader = LocalSafeDataLoader


def _memory_getitem_return_int(self, index):
    item = self.exemplars[index]
    return item.image, int(item.label)


MemoryDataset.__getitem__ = _memory_getitem_return_int
print("Local safe DataLoader aktif; seluruh label dibentuk sebagai torch.long.")


In [ ]:
# ============================================================
# 4. Unduh dan siapkan split resmi MedMNIST RetinaMNIST
# ============================================================
from torch.utils.data import Dataset

from adail.data.transforms import build_eval_transform, build_train_transform

info = medmnist.INFO[DATASET_FLAG]
DataClass = getattr(medmnist, info["python_class"])
CLASS_NAMES = [info["label"][str(i)] for i in range(NUM_CLASSES)]
idx_to_class = dict(enumerate(CLASS_NAMES))
class_to_idx = {name: idx for idx, name in idx_to_class.items()}


class RetinaMNISTDataset(Dataset):
    """Adapter agar label MedMNIST kompatibel dengan ADAIL."""

    def __init__(self, split, transform):
        self.base = DataClass(
            split=split,
            root=str(DATA_ROOT),
            transform=transform,
            download=DOWNLOAD_DATA,
            size=MEDMNIST_SIZE,
            as_rgb=True,
        )
        self.targets = [
            int(np.asarray(label).reshape(-1)[0])
            for label in self.base.labels
        ]
        self.split = split

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):
        image, label = self.base[index]
        return image, int(np.asarray(label).reshape(-1)[0])


train_dataset = RetinaMNISTDataset(
    split="train",
    transform=build_train_transform(IMAGE_SIZE),
)
val_dataset = RetinaMNISTDataset(
    split="val",
    transform=build_eval_transform(IMAGE_SIZE),
)
test_dataset = RetinaMNISTDataset(
    split="test",
    transform=build_eval_transform(IMAGE_SIZE),
)

DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset info :", info["description"])
print("Task         :", info["task"])
print("Class names  :", idx_to_class)
print("Train/val/test:", len(train_dataset), len(val_dataset), len(test_dataset))

split_rows = []
for split_name, dataset in [
    ("train", train_dataset),
    ("val", val_dataset),
    ("test", test_dataset),
]:
    counts = np.bincount(dataset.targets, minlength=NUM_CLASSES)
    for class_idx, count in enumerate(counts):
        split_rows.append(
            {
                "split": split_name,
                "class_index": class_idx,
                "class_name": idx_to_class[class_idx],
                "samples": int(count),
            }
        )

split_df = pd.DataFrame(split_rows)
split_df.to_csv(OUTPUT_DIR / "dataset_split_summary.csv", index=False)
display(split_df.pivot(index=["class_index", "class_name"], columns="split", values="samples"))


In [ ]:
# ============================================================
# 5. Training ADAIL RetinaMNIST 3-1 + LIVE HISTORY LOG
# ============================================================
import json
import time
from datetime import datetime

from torch.utils.data import ConcatDataset

from adail import ADAILConfig, ADAILNet, ADAILTrainer
from adail.data import ClassIncrementalScenario, DatasetSubsetByClass
from adail.metrics import average_accuracy, average_forgetting

LOG_JSONL = OUTPUT_DIR / "training_log.jsonl"
LOG_CSV = OUTPUT_DIR / "training_log.csv"
PARTIAL_HISTORY = OUTPUT_DIR / "history_partial.json"
FINAL_HISTORY = OUTPUT_DIR / "history.json"

for path in [LOG_JSONL, LOG_CSV, PARTIAL_HISTORY, FINAL_HISTORY]:
    if path.exists():
        path.unlink()

cfg = ADAILConfig(
    num_classes=NUM_CLASSES,
    memory_size=MEMORY_SIZE,
    use_ordinal=USE_ORDINAL,
    lambda_drift=1.0,
    lambda_proto=0.2,
    lambda_sep=0.1,
    lambda_ordinal=LAMBDA_ORDINAL,
)
cfg.data.dataset_name = DATASET_FLAG
cfg.data.root = str(DATA_ROOT)
cfg.data.scenario = SCENARIO
cfg.data.image_size = IMAGE_SIZE
cfg.data.batch_size = BATCH_SIZE
cfg.data.num_workers = NUM_WORKERS
cfg.optimizer.lr = LR
cfg.to_yaml(OUTPUT_DIR / "config.yaml")

with (OUTPUT_DIR / "class_to_idx.json").open("w", encoding="utf-8") as file:
    json.dump(class_to_idx, file, indent=2)

scenario = ClassIncrementalScenario.from_string(NUM_CLASSES, SCENARIO)
INITIAL_NUM_CLASSES = max(scenario.tasks[0]) + 1
print("CIL tasks:", scenario.tasks)
for task_idx, task_classes in enumerate(scenario.tasks):
    print(
        f"Task {task_idx + 1}: {task_classes} -> "
        f"{[idx_to_class[c] for c in task_classes]}"
    )

try:
    model = ADAILNet(
        num_classes=INITIAL_NUM_CLASSES,
        pretrained_backbone=USE_PRETRAINED_BACKBONE,
    )
except Exception as exc:
    print("ResNet-18 pretrained gagal dimuat; memakai bobot acak.")
    print("Error:", repr(exc))
    model = ADAILNet(
        num_classes=INITIAL_NUM_CLASSES,
        pretrained_backbone=False,
    )

trainer = ADAILTrainer(model=model, config=cfg, device=DEVICE)
print("Initial classifier outputs:", trainer.model.num_classes)


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")


def save_epoch_csv(records):
    if records:
        pd.DataFrame(records).to_csv(LOG_CSV, index=False)


def build_history(task_results, accuracy_matrix):
    return {
        "dataset": DATASET_FLAG,
        "scenario": SCENARIO,
        "use_ordinal": USE_ORDINAL,
        "lambda_ordinal": LAMBDA_ORDINAL,
        "task_results": task_results,
        "accuracy_matrix": accuracy_matrix,
        "average_accuracy": average_accuracy(accuracy_matrix),
        "average_forgetting": average_forgetting(accuracy_matrix),
        "class_forgetting": {
            int(k): float(v) for k, v in trainer.class_forgetting.items()
        },
        "observed_class_counts": {
            int(k): int(v) for k, v in trainer.observed_class_counts.items()
        },
    }


def fit_scenario_with_live_log(
    trainer,
    train_dataset,
    test_dataset,
    scenario,
    epochs_per_task,
    batch_size,
):
    task_accuracy_matrix = []
    task_results = []
    epoch_records = []
    total_tasks = len(scenario.tasks)

    print("\n================ ADAIL LIVE TRAINING LOG ================")
    print("Output dir :", OUTPUT_DIR)
    print("Device     :", trainer.device)
    print("Epoch/task :", epochs_per_task)
    print("Ordinal    :", USE_ORDINAL, "| lambda:", LAMBDA_ORDINAL)
    print("=========================================================")

    for task_idx, task_classes in enumerate(scenario.tasks):
        task_start = time.time()
        seen_classes = scenario.seen_classes_until(task_idx)
        trainer.current_seen_classes = seen_classes
        required_outputs = max(seen_classes) + 1

        if task_idx > 0:
            trainer.model.freeze_old_model()
            trainer.model.expand_classes(required_outputs)
        else:
            trainer.model.resize_classes(required_outputs)

        current_task_dataset = DatasetSubsetByClass(
            train_dataset,
            task_classes,
        )
        for cls, count in trainer._count_classes(current_task_dataset).items():
            trainer.observed_class_counts[cls] = (
                trainer.observed_class_counts.get(cls, 0) + count
            )

        train_available = current_task_dataset
        if len(trainer.memory) > 0:
            train_available = ConcatDataset(
                [current_task_dataset, trainer.memory.as_dataset()]
            )

        optimizer = trainer.build_optimizer()
        train_loader = adail_trainer.DataLoader(
            train_available,
            batch_size=batch_size,
            shuffle=True,
        )

        new_names = [idx_to_class[c] for c in task_classes]
        seen_names = [idx_to_class[c] for c in seen_classes]
        print(
            f"\n[Task {task_idx + 1}/{total_tasks}] "
            f"new={task_classes} {new_names}"
        )
        print("Seen classes:", seen_classes, seen_names)
        print(
            "Train current+memory:",
            len(train_available),
            "| Memory before:",
            len(trainer.memory),
        )

        epoch_logs = []
        for epoch_idx in range(epochs_per_task):
            epoch_start = time.time()
            logs = trainer.train_epoch(
                train_loader,
                optimizer,
                seen_classes=seen_classes,
            )
            elapsed = time.time() - epoch_start
            epoch_logs.append(logs)
            record = {
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "event": "epoch_end",
                "task_index": task_idx,
                "task_number": task_idx + 1,
                "total_tasks": total_tasks,
                "epoch": epoch_idx + 1,
                "epochs_per_task": epochs_per_task,
                "new_classes": str(task_classes),
                "new_class_names": str(new_names),
                "seen_classes": str(seen_classes),
                "seen_class_names": str(seen_names),
                "classifier_outputs": trainer.model.num_classes,
                "train_samples_current_plus_memory": len(train_available),
                "memory_size_before_update": len(trainer.memory),
                "lr": float(optimizer.param_groups[0]["lr"]),
                "epoch_seconds": round(elapsed, 3),
                "loss": float(logs.get("loss", 0.0)),
                "ce": float(logs.get("ce", 0.0)),
                "drift": float(logs.get("drift", 0.0)),
                "proto": float(logs.get("proto", 0.0)),
                "sep": float(logs.get("sep", 0.0)),
                "ordinal": float(logs.get("ordinal", 0.0)),
            }
            epoch_records.append(record)
            append_jsonl(LOG_JSONL, record)
            save_epoch_csv(epoch_records)
            print(
                f"[Epoch {epoch_idx + 1:02d}/{epochs_per_task}] "
                f"loss={record['loss']:.4f} ce={record['ce']:.4f} "
                f"ordinal={record['ordinal']:.4f} "
                f"drift={record['drift']:.4f} "
                f"proto={record['proto']:.4f} "
                f"sep={record['sep']:.4f} "
                f"time={record['epoch_seconds']:.1f}s"
            )

        refresh_dataset = current_task_dataset
        if len(trainer.memory) > 0:
            refresh_dataset = ConcatDataset(
                [current_task_dataset, trainer.memory.as_dataset()]
            )
        refresh_loader = adail_trainer.DataLoader(
            refresh_dataset,
            batch_size=batch_size,
            shuffle=False,
        )
        trainer.refresh_prototypes(refresh_loader)

        current_loader = adail_trainer.DataLoader(
            current_task_dataset,
            batch_size=batch_size,
            shuffle=False,
        )
        trainer.update_memory(current_loader)

        row = []
        metrics_by_task = {}
        for eval_task_idx in range(task_idx + 1):
            eval_classes = scenario.tasks[eval_task_idx]
            eval_loader = adail_trainer.DataLoader(
                DatasetSubsetByClass(test_dataset, eval_classes),
                batch_size=batch_size,
                shuffle=False,
            )
            eval_metrics = trainer.evaluate(
                eval_loader,
                seen_classes=seen_classes,
            )
            row.append(float(eval_metrics["accuracy"]))
            for metric_name, metric_value in eval_metrics.items():
                if isinstance(metric_value, float):
                    metrics_by_task[
                        f"task_{eval_task_idx}_{metric_name}"
                    ] = float(metric_value)
        task_accuracy_matrix.append(row)

        seen_loader = adail_trainer.DataLoader(
            DatasetSubsetByClass(test_dataset, seen_classes),
            batch_size=batch_size,
            shuffle=False,
        )
        seen_metrics = trainer.evaluate(
            seen_loader,
            seen_classes=seen_classes,
        )
        class_scores = seen_metrics.get("per_class_accuracy", {})
        trainer._update_forgetting_estimates(class_scores)

        metrics_by_task.update(
            {
                "seen_accuracy": float(seen_metrics["accuracy"]),
                "seen_mae": float(seen_metrics["mae"]),
                "seen_qwk": float(seen_metrics["qwk"]),
                "last_loss": float(epoch_logs[-1]["loss"])
                if epoch_logs
                else 0.0,
            }
        )
        task_result = {
            "task_index": task_idx,
            "new_classes": list(map(int, task_classes)),
            "new_class_names": new_names,
            "seen_classes": list(map(int, seen_classes)),
            "seen_class_names": seen_names,
            "metrics": metrics_by_task,
            "per_class_accuracy": {
                int(k): float(v) for k, v in class_scores.items()
            },
            "memory_quotas": {
                int(k): int(v) for k, v in trainer.memory.quotas.items()
            },
            "memory_size_after_update": int(len(trainer.memory)),
            "task_seconds": round(time.time() - task_start, 3),
        }
        task_results.append(task_result)

        task_event = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "event": "task_end",
            "task_index": task_idx,
            "task_number": task_idx + 1,
            "seen_accuracy": metrics_by_task["seen_accuracy"],
            "seen_mae": metrics_by_task["seen_mae"],
            "seen_qwk": metrics_by_task["seen_qwk"],
            "memory_size_after_update": len(trainer.memory),
            "memory_quotas": str(task_result["memory_quotas"]),
            "accuracy_row": str(row),
            "task_seconds": task_result["task_seconds"],
        }
        append_jsonl(LOG_JSONL, task_event)

        partial = build_history(task_results, task_accuracy_matrix)
        PARTIAL_HISTORY.write_text(
            json.dumps(partial, indent=2),
            encoding="utf-8",
        )
        FINAL_HISTORY.write_text(
            json.dumps(partial, indent=2),
            encoding="utf-8",
        )

        print(
            f"[Task {task_idx + 1} selesai] "
            f"accuracy={metrics_by_task['seen_accuracy']:.4f} "
            f"MAE={metrics_by_task['seen_mae']:.4f} "
            f"QWK={metrics_by_task['seen_qwk']:.4f} "
            f"memory={len(trainer.memory)}"
        )

    final_history = build_history(task_results, task_accuracy_matrix)
    FINAL_HISTORY.write_text(
        json.dumps(final_history, indent=2),
        encoding="utf-8",
    )
    save_epoch_csv(epoch_records)
    return final_history


history = fit_scenario_with_live_log(
    trainer=trainer,
    train_dataset=train_dataset,
    test_dataset=test_dataset,
    scenario=scenario,
    epochs_per_task=EPOCHS_PER_TASK,
    batch_size=BATCH_SIZE,
)

CHECKPOINT_PATH = OUTPUT_DIR / "adail_retinamnist_last.pt"
trainer.save_checkpoint(CHECKPOINT_PATH)

print("\n================ FINAL CIL RESULT ================")
print("Average accuracy  :", history["average_accuracy"])
print("Average forgetting:", history["average_forgetting"])
print("History            :", FINAL_HISTORY)
print("Checkpoint         :", CHECKPOINT_PATH)
print("==================================================")


In [ ]:
# ============================================================
# 6. Evaluasi ordinal akhir pada seluruh test set
# ============================================================
from adail.metrics import (
    mean_absolute_error,
    quadratic_weighted_kappa,
)

all_seen_classes = list(range(NUM_CLASSES))
test_loader = adail_trainer.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

trainer.model.eval()
all_labels = []
all_predictions = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(trainer.device)
        logits = trainer.model(images)
        logits = trainer._mask_logits(logits, all_seen_classes)
        predictions = logits.argmax(dim=1).cpu()
        all_labels.append(labels.cpu())
        all_predictions.append(predictions)

labels_np = torch.cat(all_labels).numpy().astype(int)
preds_np = torch.cat(all_predictions).numpy().astype(int)
absolute_errors = np.abs(preds_np - labels_np)

ordinal_metrics = {
    "samples": int(len(labels_np)),
    "accuracy": float(np.mean(preds_np == labels_np)),
    "mae": mean_absolute_error(preds_np, labels_np),
    "qwk": quadratic_weighted_kappa(
        preds_np,
        labels_np,
        num_classes=NUM_CLASSES,
    ),
    "within_one_grade_accuracy": float(np.mean(absolute_errors <= 1)),
    "mean_squared_grade_error": float(np.mean(absolute_errors ** 2)),
}

prediction_df = pd.DataFrame(
    {
        "true_label": labels_np,
        "true_class": [idx_to_class[int(x)] for x in labels_np],
        "predicted_label": preds_np,
        "predicted_class": [idx_to_class[int(x)] for x in preds_np],
        "absolute_error": absolute_errors,
    }
)
prediction_df.to_csv(OUTPUT_DIR / "ordinal_predictions.csv", index=False)

confusion = pd.crosstab(
    pd.Categorical(labels_np, categories=range(NUM_CLASSES)),
    pd.Categorical(preds_np, categories=range(NUM_CLASSES)),
    rownames=["true_label"],
    colnames=["predicted_label"],
    dropna=False,
)
confusion.index = [f"{i}: {idx_to_class[i]}" for i in range(NUM_CLASSES)]
confusion.columns = [f"{i}: {idx_to_class[i]}" for i in range(NUM_CLASSES)]
confusion.to_csv(OUTPUT_DIR / "ordinal_confusion_matrix.csv")

(OUTPUT_DIR / "ordinal_evaluation.json").write_text(
    json.dumps(ordinal_metrics, indent=2),
    encoding="utf-8",
)

print("Ordinal evaluation:")
for name, value in ordinal_metrics.items():
    print(f"- {name}: {value}")
display(confusion)
display(prediction_df.head(20))


In [ ]:
# ============================================================
# 7. Ringkasan hasil per task dan accuracy matrix
# ============================================================
history = json.loads(FINAL_HISTORY.read_text(encoding="utf-8"))

summary_rows = []
for task in history.get("task_results", []):
    summary_rows.append(
        {
            "task_index": task.get("task_index"),
            "new_classes": task.get("new_classes"),
            "new_class_names": task.get("new_class_names"),
            "seen_classes": task.get("seen_classes"),
            "seen_accuracy": task.get("metrics", {}).get("seen_accuracy"),
            "seen_mae": task.get("metrics", {}).get("seen_mae"),
            "seen_qwk": task.get("metrics", {}).get("seen_qwk"),
            "last_loss": task.get("metrics", {}).get("last_loss"),
            "memory_size_after_update": task.get(
                "memory_size_after_update"
            ),
            "memory_quotas": task.get("memory_quotas"),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "summary_by_task.csv", index=False)
display(summary_df)

accuracy_matrix = history.get("accuracy_matrix", [])
max_columns = max((len(row) for row in accuracy_matrix), default=0)
accuracy_df = pd.DataFrame(
    [
        row + [None] * (max_columns - len(row))
        for row in accuracy_matrix
    ],
    columns=[f"eval_task_{i + 1}" for i in range(max_columns)],
)
accuracy_df.index = [
    f"after_task_{i + 1}" for i in range(len(accuracy_df))
]
accuracy_df.to_csv(OUTPUT_DIR / "accuracy_matrix.csv")
display(accuracy_df)

if LOG_CSV.exists():
    display(pd.read_csv(LOG_CSV).tail(20))

print("Average accuracy  :", history.get("average_accuracy"))
print("Average forgetting:", history.get("average_forgetting"))


In [ ]:
# ============================================================
# 8. Buat ZIP seluruh output
# ============================================================
import shutil

ZIP_BASE = OUTPUT_DIR.parent / "adail_retinamnist_outputs"
zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=OUTPUT_DIR,
)

print("ZIP output:", zip_path)
print("Files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path.name)
